# 🧬 TCGA Pan-Cancer RNA-Seq Classification Pipeline
### High-Dimensional Multi-Class Transcriptomic Stratification & Explainable AI

**Clinical Overview:**
This research pipeline addresses the challenge of multi-class cancer classification from high-dimensional RNA-Seq gene expression data (=20,531$ genes across =801$ patients) from The Cancer Genome Atlas (TCGA).

**Target Malignancies (5 Cohorts):**
1. **BRCA**: Breast Invasive Carcinoma (300 patients)
2. **KIRC**: Kidney Renal Clear Cell Carcinoma (146 patients)
3. **LUAD**: Lung Adenocarcinoma (141 patients)
4. **PRAD**: Prostate Adenocarcinoma (136 patients)
5. **COAD**: Colon Adenocarcinoma (78 patients)

**Key Methodological Safeguards:**
- **Zero Selection Leakage:** Preprocessing and feature selection are embedded inside Scikit-Learn pipelines to ensure CV validation folds remain strictly blind.
- **Stratified Splitting:** Preserves cohort proportions across train (0\%$) and held-out test (0\%$) sets.
- **Multi-Model Benchmark:** Benchmarks Linear SVM, Multinomial Logistic Regression, Random Forest, and XGBoost.
- **Explainable AI (XAI):** Local logit attributions and global ANOVA biomarker mining.

In [ ]:
import os
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, roc_auc_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
print("✓ Core genomic data science libraries loaded.")

## 1. Data Ingestion & Exploratory Cohort Analysis

In [ ]:
X = pd.read_csv("dataset/data.csv", index_col=0)
y = pd.read_csv("dataset/labels.csv", index_col=0).iloc[:, 0]

le = LabelEncoder()
y_encoded = le.fit_transform(y)
classes = list(le.classes_)

print(f"Dataset Dimensions: {X.shape[0]} patients, {X.shape[1]} RNA-Seq features")
print("Cohort Breakdown:")
print(y.value_counts())

## 2. Partitioning (Strict 80/20 Stratified Split)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f"Training Cohort: {X_train.shape[0]} samples")
print(f"Held-Out Test Cohort: {X_test.shape[0]} samples")

## 3. Multi-Model Cross-Validation Benchmark (Zero Selection Leakage)
Each model is evaluated with an integrated Pipeline containing  $	o$  $	o$ .

In [ ]:
models = {
    "Logistic_Regression": LogisticRegression(C=0.1, max_iter=1000, class_weight="balanced", random_state=42),
    "Linear_SVM": CalibratedClassifierCV(SVC(kernel="linear", C=0.01, random_state=42), ensemble=False),
    "Random_Forest": RandomForestClassifier(n_estimators=100, max_depth=15, class_weight="balanced", random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, eval_metric="mlogloss", random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, clf in models.items():
    pipe = Pipeline([
        ("var", VarianceThreshold(0.1)),
        ("scaler", StandardScaler()),
        ("select", SelectKBest(f_classif, k=500)),
        ("clf", clf)
    ])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=["accuracy", "f1_macro"], n_jobs=-1)
    acc = scores["test_accuracy"].mean()
    f1 = scores["test_f1_macro"].mean()
    cv_results[name] = {"CV Accuracy": acc, "CV F1-Macro": f1}
    print(f"{name:20s} | CV Accuracy: {acc*100:.2f}% | CV F1-Macro: {f1*100:.2f}%")

pd.DataFrame(cv_results).T

## 4. Final Model Fitting & Independent Test Evaluation

In [ ]:
champion_pipe = Pipeline([
    ("var", VarianceThreshold(0.1)),
    ("scaler", StandardScaler()),
    ("select", SelectKBest(f_classif, k=500)),
    ("clf", LogisticRegression(C=0.1, max_iter=1000, class_weight="balanced", random_state=42))
])

champion_pipe.fit(X_train, y_train)
y_test_pred = champion_pipe.predict(X_test)
y_test_prob = champion_pipe.predict_proba(X_test)

print("=== Classification Report ===")
print(classification_report(y_test, y_test_pred, target_names=classes))
print(f"Held-Out Test Accuracy: {accuracy_score(y_test, y_test_pred)*100:.2f}%")
print(f"Held-Out Test ROC-AUC (OvR): {roc_auc_score(y_test, y_test_prob, multi_class="ovr")*100:.2f}%")

# Confusion Matrix Heatmap
plt.figure(figsize=(7, 5))
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.title("Held-Out Test Confusion Matrix (N=161)")
plt.ylabel("True Cancer Cohort")
plt.xlabel("Predicted Cancer Cohort")
plt.tight_layout()
plt.show()

## 5. 3D Principal Component Analysis (PCA) Manifold

In [ ]:
X_train_prep = champion_pipe[:-1].transform(X_train)
pca = PCA(n_components=3, random_state=42)
coords = pca.fit_transform(X_train_prep)
ev = pca.explained_variance_ratio_
print(f"PCA Explained Variance: PC1={ev[0]*100:.1f}%, PC2={ev[1]*100:.1f}%, PC3={ev[2]*100:.1f}%")

## 6. Explainable AI: Feature Attributions & Biomarker Mining

In [ ]:
var_mask = champion_pipe.named_steps["var"].get_support()
select_mask = champion_pipe.named_steps["select"].get_support()
selected_genes = X.columns[var_mask][select_mask]
f_scores = champion_pipe.named_steps["select"].scores_[select_mask]

biomarker_df = pd.DataFrame({
    "Gene_ID": selected_genes,
    "ANOVA_F_Score": f_scores
}).sort_values("ANOVA_F_Score", ascending=False)

print("Top 10 Universal Biomarkers by ANOVA F-Score:")
print(biomarker_df.head(10))